In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ファイルパス（先ほどリネームして保存した名前）
file_path = "../data/raw/cicids/02-14-2018.csv"

# 読み込み（400MB弱あるので、数秒かかります）
# low_memory=False を指定して型推論の警告を回避します
df = pd.read_csv(file_path, low_memory=False)

# カラム名にスペースが含まれている場合があるため、トリミングしておく
df.columns = df.columns.str.strip()

print(f"データの総行数: {len(df):,}")
print("-" * 30)
# 最初の5行を表示
display(df.head())

データの総行数: 1,048,575
------------------------------


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,14/02/2018 08:31:01,112641719,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign
1,0,0,14/02/2018 08:33:50,112641466,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign
2,0,0,14/02/2018 08:36:39,112638623,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign
3,22,6,14/02/2018 08:40:13,6453966,15,10,1239,2273,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
4,22,6,14/02/2018 08:40:23,8804066,14,11,1143,2209,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign


In [ ]:
# ラベルごとの件数をカウント
label_counts = df['Label'].value_counts()
print("ラベルの内訳:")
print(label_counts)

# 棒グラフで可視化
plt.figure(figsize=(10, 6))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis')
plt.title('Distribution of Attack Labels (2018-02-14)')
plt.xlabel('Label')
plt.ylabel('Count (log scale)')
plt.yscale('log') # 件数差が激しいため対数スケールにすると見やすいです
plt.xticks(rotation=45)
plt.show()

In [5]:
import pandas as pd
import numpy as np

# --- ステップ1: 掃除 ---
df_cleaned = df.replace([np.inf, -np.inf], np.nan).dropna()

# --- ステップ2: ベクトルDB用の「知識」を抽出 (正常のみ) ---
benign_pool = df_cleaned[df_cleaned['Label'] == 'Benign']
knowledge_df = benign_pool.sample(n=100, random_state=42)

# --- ステップ3: 現実的な「テストデータ」を作成 ---
# 1. 攻撃ログのプールを定義（ここで定義！）
attack_pool = df_cleaned[df_cleaned['Label'] != 'Benign']

# 2. 正常ログ（知識用に使わなかった残り）から 95件
test_benign = benign_pool.drop(knowledge_df.index).sample(n=95, random_state=7)

# 3. 攻撃ログから 5件
test_attack = attack_pool.sample(n=5, random_state=7)

# 4. 結合してシャッフル
test_data_df = pd.concat([test_benign, test_attack]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"知識データ（DB用）: {len(knowledge_df)}件")
print(f"テストデータ合計: {len(test_data_df)}件")
print("テストデータ内の内訳:\n", test_data_df['Label'].value_counts())

知識データ（DB用）: 100件
テストデータ合計: 100件
テストデータ内の内訳:
 Label
Benign            95
FTP-BruteForce     3
SSH-Bruteforce     2
Name: count, dtype: int64


In [12]:
import os

# 保存先のディレクトリを作成（まだ存在しない場合）
output_dir = "../data/processed/cicids"
os.makedirs(output_dir, exist_ok=True)

# 1. 知識用データ（正常100件）を保存
knowledge_path = os.path.join(output_dir, "knowledge_benign_100.csv")
knowledge_df.to_csv(knowledge_path, index=False)

# 2. テスト用データ（不均衡100件）を保存
test_data_path = os.path.join(output_dir, "test_imbalanced_100.csv")
test_data_df.to_csv(test_data_path, index=False)

print(f"保存完了:\n- {knowledge_path}\n- {test_data_path}")

保存完了:
- ../data/processed/cicids/knowledge_benign_100.csv
- ../data/processed/cicids/test_imbalanced_100.csv
